### Import dependencies

In [2]:
from ultralytics import YOLO
import cv2
import os
from pathlib import Path

### Specify trained YOLO model
All models are saved to runs/detect/train-xx/weights/best.pt

In [10]:
model = YOLO("best.pt")

### Run a video through the model with predict()
Output is a video with bounding boxes around detected objects with conf score on the outside. NO tracking yet. Results saved to runs/detect/predict-xx

In [11]:
# Specify video path
video_path = "/Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4"

In [12]:
results = model.predict(
    source=video_path,
    conf=0.15, iou=0.2,
    save=True, device='mps', stream=True
)

for result in results:
    pass


video 1/1 (frame 1/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 384x640 (no detections), 366.9ms
video 1/1 (frame 2/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 384x640 (no detections), 43.7ms
video 1/1 (frame 3/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 384x640 (no detections), 75.4ms
video 1/1 (frame 4/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 384x640 (no detections), 64.4ms
video 1/1 (frame 5/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 384x640 (no detections), 41.5ms
video 1/1 (frame 6/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 384x640 (no detections), 40.0ms
video 1/1 (frame 7/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 384x640 (no detections), 60.0ms
vide

Same thing for model.track() but results will include ID assigned to unique organisms.

### Capture images of unique organisms with tracking after detection

Create output directory

In [ ]:
os.makedirs('./tracked-frames/', exist_ok=True)
output_folder = Path('tracked-frames')  / video_path.stem
output_folder.mkdir(parents=True, exist_ok=True)

Loop through tracked frames to get labels and draw bounding boxes on an image copy. Output 1 image for each unique object

In [ ]:

best_frames = {}
results = model.track(source=video_path, device='mps', stream=True, persist=True, tracker='botsort.yaml', conf=0.15)

for frame_index, result in enumerate(results):
    if result.boxes is None or result.boxes.id is None:
        continue
    print(f"frame {frame_index}: {len(result.boxes)} detections")
    raw_frame = result.orig_img
    boxes = result.boxes.xyxy.cpu().numpy()
    tracking_ids = result.boxes.id.cpu().numpy().astype(int)
    confidences = result.boxes.conf.cpu().numpy()

    for box, track_id, conf in zip(boxes, tracking_ids, confidences):
        x1, y1, x2, y2 = map(int, box)
        if track_id not in best_frames:
            best_frames[track_id] = {
                'highest_conf': conf,
                'frame_matrix': raw_frame.copy(),
                'box_coords': (x1,y1,x2,y2)
            }       

        else:
            if conf > best_frames[track_id]['highest_conf']:
                best_frames[track_id]['highest_conf'] = conf
                best_frames[track_id]['frame_matrix'] = raw_frame.copy()
                best_frames[track_id]['box_coords'] = (x1,y1,x2,y2)

for track_id, data in best_frames.items():
    full_frame = data['frame_matrix']
    x1,y1,x2,y2 = data['box_coords']
    conf_score = data['highest_conf']

    box_color = (0,255,0)
    label_text = f"ID: {track_id} ({conf_score:.2f})"
    
    # Draw the rectangle on the full image copy
    cv2.rectangle(full_frame, (x1, y1), (x2, y2), box_color, thickness=3)
    
    # Add the text label right above the box
    cv2.putText(
        full_frame, 
        label_text, 
        (x1, max(y1 - 10, 20)), 
        fontFace=cv2.FONT_HERSHEY_SIMPLEX, 
        fontScale=0.7, 
        color=box_color, 
        thickness=2,
        lineType=cv2.LINE_AA)
    
    filename = f"organism_id_{track_id}_{frame_index}.jpg"
    save_path = os.path.join(output_folder, filename)
    cv2.imwrite(save_path, full_frame)


print(f"Done! Successfully generated {len(best_frames)} unique organism images.")


video 1/1 (frame 1/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 736x1280 (no detections), 32.9ms
video 1/1 (frame 2/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 736x1280 (no detections), 22.3ms
video 1/1 (frame 3/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 736x1280 (no detections), 23.9ms
video 1/1 (frame 4/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 736x1280 (no detections), 23.5ms
WARNING ⚠️ not enough matching points
video 1/1 (frame 5/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 736x1280 (no detections), 22.9ms
WARNING ⚠️ not enough matching points
video 1/1 (frame 6/1798) /Users/alopias/Desktop/Huyen-deePi/test_converted_videos/10.0.16.2_202309020548.mp4: 736x1280 (no detections), 24.4ms
WARNING ⚠️ not enough matching points
video 1/1 (frame 7/1798) /U